In [1]:
# Import required libraries
import pandas as pd
import numpy as np

# Step 1: Load data using chunking
pickup_chunks = pd.read_csv("/content/drive/MyDrive/data/cleaned_pickup_data.csv", chunksize=50000)
delivery_chunks = pd.read_csv("/content/drive/MyDrive/data/cleaned_delivery_data_edit.csv", chunksize=50000)

# Combine chunks for Pickup and Delivery datasets
pickup_data = pd.concat(pickup_chunks, ignore_index=True)
delivery_data = pd.concat(delivery_chunks, ignore_index=True)

# Step 2: Inspect datasets
print("Pickup Dataset Info:")
print(pickup_data.info())
print("\nDelivery Dataset Info:")
print(delivery_data.info())

# Check for duplicates
pickup_data = pickup_data.drop_duplicates(subset="order_id")
delivery_data = delivery_data.drop_duplicates(subset="order_id")

# Step 3: Merge the datasets on 'order_id' (inner join)
merged_data = pd.merge(pickup_data, delivery_data, on="order_id", how="inner")



# Step 4: Calculate the ETA (in minutes)
merged_data["pickup_time"] = pd.to_datetime(merged_data["pickup_time"])
merged_data["delivery_time"] = pd.to_datetime(merged_data["delivery_time"])
merged_data["ETA"] = (merged_data["delivery_time"] - merged_data["pickup_time"]).dt.total_seconds() / 60

# Step 5: Handle missing values
# Drop rows with missing pickup_time, delivery_time, or coordinates
merged_data = merged_data.dropna(subset=["pickup_time", "delivery_time", "pickup_gps_lng", "pickup_gps_lat", "delivery_gps_lng", "delivery_gps_lat"])

# Step 6: Save the merged dataset
merged_data.to_csv("merged_datasett.csv", index=False)
print(merged_data)


Pickup Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6136147 entries, 0 to 6136146
Data columns (total 22 columns):
 #   Column                Dtype  
---  ------                -----  
 0   order_id              int64  
 1   region_id             int64  
 2   city                  object 
 3   courier_id            int64  
 4   accept_time           object 
 5   time_window_start     object 
 6   time_window_end       object 
 7   lng                   float64
 8   lat                   float64
 9   aoi_id                int64  
 10  aoi_type              int64  
 11  pickup_time           object 
 12  pickup_gps_time       object 
 13  pickup_gps_lng        float64
 14  pickup_gps_lat        float64
 15  accept_gps_time       object 
 16  accept_gps_lng        float64
 17  accept_gps_lat        float64
 18  ds                    int64  
 19  time_window_duration  float64
 20  task_duration         float64
 21  distance              float64
dtypes: float64(9), int64(

In [2]:
print(merged_data["ETA"].describe())

count    3.247628e+06
mean     6.523268e+07
std      1.050945e+05
min      6.482606e+07
25%      6.515906e+07
50%      6.523316e+07
75%      6.530986e+07
max      6.551109e+07
Name: ETA, dtype: float64


In [3]:

merged_data = merged_data[merged_data["ETA"] >= 0]

print(merged_data["ETA"].describe())

count    3.247628e+06
mean     6.523268e+07
std      1.050945e+05
min      6.482606e+07
25%      6.515906e+07
50%      6.523316e+07
75%      6.530986e+07
max      6.551109e+07
Name: ETA, dtype: float64


In [2]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
merge_chunks = pd.read_csv("/content/merged_datasett.csv", chunksize=50000)

# Combine chunks for Pickup and Delivery datasets
merged_data = pd.concat(merge_chunks, ignore_index=True)

# Process the data in chunks
chunk_size = 50_000  # Number of rows to process at a time
file_path = "/content/drive/MyDrive/data/roads.csv"

# Iterate through chunks
for road_df in pd.read_csv(file_path,delimiter='\t', chunksize=chunk_size):
    # Perform your processing here
    print(road_df.shape)

road_df.head()
    # Perform your processing here
print(road_df.shape)

road_df.head()

# Convert pickup and delivery coordinates to GeoDataFrame
merged_data['pickup_point'] = merged_data.apply(lambda row: Point(row['pickup_gps_lng'], row['pickup_gps_lat']), axis=1)
merged_data['delivery_point'] = merged_data.apply(lambda row: Point(row['delivery_gps_lng'], row['delivery_gps_lat']), axis=1)

# Convert merged_df to GeoDataFrame
merged_gdf = gpd.GeoDataFrame(merged_data, geometry='pickup_point')

(50000, 12)
(50000, 12)
(50000, 12)
(50000, 12)
(50000, 12)
(50000, 12)
(50000, 12)
(50000, 12)
(50000, 12)
(50000, 12)
(31280, 12)
(31280, 12)


In [6]:
road_df.columns
road_df.dtypes

,0
osm_id,int64
code,int64
fclass,object
name,object
ref,object
oneway,object
maxspeed,int64
layer,int64
bridge,object
tunnel,object


In [3]:
# Convert WKT strings to Shapely geometries
from shapely import wkt
road_df['geometry'] = road_df['geometry'].apply(wkt.loads)

In [4]:
# Convert to GeoDataFrame
road_df = gpd.GeoDataFrame(road_df, geometry='geometry')

In [5]:

# Convert pickup and delivery coordinates to GeoDataFrame
merged_data['pickup_point'] = merged_data.apply(lambda row: Point(row['pickup_gps_lng'], row['pickup_gps_lat']), axis=1)
merged_data['delivery_point'] = merged_data.apply(lambda row: Point(row['delivery_gps_lng'], row['delivery_gps_lat']), axis=1)

pickup_gdf = gpd.GeoDataFrame(merged_data, geometry='pickup_point')
delivery_gdf = gpd.GeoDataFrame(merged_data, geometry='delivery_point')

# Perform spatial joins for pickup and delivery points
pickup_roads = gpd.sjoin_nearest(
    pickup_gdf[['order_id', 'pickup_point']],
    road_df[['geometry', 'maxspeed', 'fclass', 'oneway', 'tunnel', 'bridge']],
    how='left',
    distance_col='pickup_distance'
)

delivery_roads = gpd.sjoin_nearest(
    delivery_gdf[['order_id', 'delivery_point']],
    road_df[['geometry', 'maxspeed', 'fclass', 'oneway', 'tunnel', 'bridge']],
    how='left',
    distance_col='delivery_distance'
)

# Rename columns for clarity
pickup_roads = pickup_roads.rename(columns={
    'maxspeed': 'pickup_maxspeed',
    'fclass': 'pickup_fclass',
    'oneway': 'pickup_oneway',
    'tunnel': 'pickup_tunnel',
    'bridge': 'pickup_bridge',
    'pickup_distance': 'pickup_road_distance'
})

delivery_roads = delivery_roads.rename(columns={
    'maxspeed': 'delivery_maxspeed',
    'fclass': 'delivery_fclass',
    'oneway': 'delivery_oneway',
    'tunnel': 'delivery_tunnel',
    'bridge': 'delivery_bridge',
    'delivery_distance': 'delivery_road_distance'
})

In [ ]:

# Perform the merge with pickup_roads
merged_with_roads = pd.merge(merged_data, pickup_roads, on='order_id', how='left')
merged_with_roads = pd.merge(merged_with_roads, delivery_roads, on='order_id', how='left')
from geopy.distance import geodesic

# Calculate road-based distance using geodesic (Haversine)
merged_with_roads['road_distance'] = merged_with_roads.apply(
    lambda row: geodesic((row['pickup_gps_lat'], row['pickup_gps_lng']), (row['delivery_gps_lat'], row['delivery_gps_lng'])).km,
    axis=1
)

# Display dataset with road distance
print("Dataset with Road Distance:")
print(merged_with_roads.head(5))

# Convert columns to numeric, coercing errors to NaN
merged_with_roads['pickup_maxspeed'] = pd.to_numeric(merged_with_roads['pickup_maxspeed'], errors='coerce')
merged_with_roads['delivery_maxspeed'] = pd.to_numeric(merged_with_roads['delivery_maxspeed'], errors='coerce')
merged_with_roads['pickup_tunnel'] = pd.to_numeric(merged_with_roads['pickup_tunnel'], errors='coerce')
merged_with_roads['delivery_tunnel'] = pd.to_numeric(merged_with_roads['delivery_tunnel'], errors='coerce')
merged_with_roads['pickup_bridge'] = pd.to_numeric(merged_with_roads['pickup_bridge'], errors='coerce')
merged_with_roads['delivery_bridge'] = pd.to_numeric(merged_with_roads['delivery_bridge'], errors='coerce')

# Fill NaN values with 0 (or another default value)
merged_with_roads['pickup_maxspeed'] = merged_with_roads['pickup_maxspeed'].fillna(0)
merged_with_roads['delivery_maxspeed'] = merged_with_roads['delivery_maxspeed'].fillna(0)
merged_with_roads['pickup_tunnel'] = merged_with_roads['pickup_tunnel'].fillna(0)
merged_with_roads['delivery_tunnel'] = merged_with_roads['delivery_tunnel'].fillna(0)
merged_with_roads['pickup_bridge'] = merged_with_roads['pickup_bridge'].fillna(0)
merged_with_roads['delivery_bridge'] = merged_with_roads['delivery_bridge'].fillna(0)

# Check data types after conversion
print("Data types after conversion:")
print(merged_with_roads[['pickup_maxspeed', 'delivery_maxspeed', 'pickup_tunnel', 'delivery_tunnel', 'pickup_bridge', 'delivery_bridge']].dtypes)

# Calculate average speed limit
merged_with_roads['avg_speed_limit'] = (
    merged_with_roads['pickup_maxspeed'] +
    merged_with_roads['delivery_maxspeed']
) / 2

# Calculate proportion of tunnels and bridges
merged_with_roads['tunnel_proportion'] = (
    merged_with_roads['pickup_tunnel'] +
    merged_with_roads['delivery_tunnel']
) / 2
merged_with_roads['bridge_proportion'] = (
    merged_with_roads['pickup_bridge'] +
    merged_with_roads['delivery_bridge']
) / 2
# Save dataset with road features
merged_with_roads.to_csv('merged_with_roads.csv', index=False)
print("Dataset with road features saved as 'merged_with_roads.csv'.")
